# Explorer: any predictor or structural metric, multipass vs every other class

Set the parameters in the next cell and run the notebook (or use the dropdowns at the end if `ipywidgets` is installed).

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Markdown, display
sys.path.insert(0, str(Path.cwd().parent))          # membrane_analysis/ (when not pip-installed)
warnings.filterwarnings("ignore", category=RuntimeWarning)
from mvsa import analyses as A, plots as P, report as R, stats as S
from mvsa.data import load, available_predictors
from mvsa.groups import CLASS_ORDER, CLASS_LABEL, CLASS_COLOR, MEMBRANE, MAIN_PREDICTORS, PREDICTOR_GROUP, DIRECTION

# ---------------------------------------------------------------- parameters
DATA_DIR = Path(os.environ.get("MVSA_DATA", "../data"))     # written by `python -m mvsa.prepare`
FIG_DIR = Path(os.environ.get("MVSA_FIGURES", "../figures"))
INCLUDE_LIKELY = True       # count likely pathogenic / likely benign as labels
MIN_STARS = 0               # ClinVar review filter: 0 all, 1 drop 'no assertion criteria', 2 multiple submitters+
N_BOOT = int(os.environ.get("MVSA_NBOOT", 300))   # protein bootstrap resamples for CIs
MIN_PER_CLASS = 5           # per-protein AUCs need this many pathogenic AND benign variants
HUMAN = 9606

P.setup(FIG_DIR)
D = load(DATA_DIR)
prot = D["proteins"]
lab_all = D["labelled_variants"]
lab = S.labels(lab_all, include_likely=INCLUDE_LIKELY, min_stars=MIN_STARS)     # pathogenic (y=1) / benign (y=0)
lab = lab[lab.protein_class.notna()]
sample = D["variant_sample"]
PRED = [p for p in MAIN_PREDICTORS if p in available_predictors(lab)]
VEP = [p for p in ("alphamissense", "esm1v", "saprot") if p in PRED]
print("data:", D["meta"].get("created", "?"), "| predictors with data:", ", ".join(PRED))
missing = [p for p in MAIN_PREDICTORS if p not in PRED]
if missing:
    print("no data yet (skipped):", ", ".join(missing))

# ---------------------------------------------------------------- explorer choices
PREDICTOR = 'alphamissense'          # any of: 'alphamissense', 'esm1v', 'saprot', 'thermompnn', 'foldx', 'blosum62', 'abs_delta_hydrophobicity', 'abs_delta_volume', 'abs_delta_charge', 'delta_hydrophobicity', 'delta_volume', 'delta_charge', 'solvent_exposure'
METRIC = 'a_over_c'                  # any column of shape_units (see SHAPE_METRICS below)
UNIT = 'CHAIN_TRIMMED'               # CHAIN_FULL, CHAIN_TRIMMED, DOMAIN, TM_CORE
CLASSES = CLASS_ORDER
GENE = 'CFTR'                        # protein lookup
SHAPE_METRICS = [c for c in D['shape_units'].columns if pd.api.types.is_numeric_dtype(D['shape_units'][c])]

## Predictor view

In [ ]:
def predictor_view(p):
    if p not in lab or lab[p].notna().sum() == 0:
        print(p, 'has no data'); return
    pooled = A.pooled_performance(lab, [p], n_boot=min(N_BOOT, 200))
    loc = A.location_performance(lab, [p], n_boot=100)
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    P.forest(pooled[pooled['class'].isin(CLASSES)], 'class', 'class', ax=axes[0, 0], groups=[c for c in CLASSES], title='Pooled AUC')
    for c, (fpr, tpr, a) in A.roc_curves(lab, p, CLASSES).items():
        axes[0, 1].plot(fpr, tpr, color=CLASS_COLOR[c], label=f'{CLASS_LABEL[c]} {a:.2f}')
    axes[0, 1].plot([0, 1], [0, 1], color=P.AXIS); axes[0, 1].legend(fontsize=7); axes[0, 1].set_title('ROC', loc='left')
    pp, _ = A.per_protein_performance(D['protein_auc'], [p], MIN_PER_CLASS)
    P.class_violin(pp, 'auc', CLASSES, ax=axes[0, 2], ylabel='per-protein AUC', ref=0.5, title='Per protein')
    P.ecdf(sample.assign(v=sample[p] * DIRECTION[p]), 'v', CLASSES, ax=axes[1, 0], xlabel='oriented score (all substitutions)')
    l = loc.reset_index(drop=True)
    axes[1, 1].errorbar(l.auc, range(len(l)), xerr=[l.auc - l.ci_low, l.ci_high - l.auc], fmt='o', color=P.INK2)
    axes[1, 1].set_yticks(range(len(l)), l['class'].str.title() + ': ' + l.location, fontsize=7); axes[1, 1].axvline(0.5, color=P.AXIS)
    axes[1, 1].invert_yaxis(); axes[1, 1].set_title('AUC by location', loc='left')
    calib, curves = A.calibration(lab, [p])
    for c in CLASSES:
        r = curves[curves['class'] == c] if len(curves) else curves
        if len(r): axes[1, 2].plot(r.pred, r.obs, 'o-', color=CLASS_COLOR[c], ms=3, label=CLASS_LABEL[c])
    axes[1, 2].plot([0, 1], [0, 1], color=P.AXIS); axes[1, 2].set_title('Calibration (fitted on soluble)', loc='left'); axes[1, 2].legend(fontsize=7)
    fig.suptitle(P.label(p), x=0.01, ha='left'); fig.tight_layout(); plt.show()
    display(pooled[['class', 'n_proteins', 'n_pathogenic', 'n_benign', 'auc', 'ci_low', 'ci_high', 'mp_minus_delta', 'mp_minus_q']].round(3))

predictor_view(PREDICTOR)

## Structural metric view

In [ ]:
def metric_view(metric, unit):
    sh = D['shape_units']; d = sh[sh.unit == unit]
    if metric not in d: print(metric, 'not available'); return
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    P.class_violin(d, metric, CLASSES, ax=axes[0], title=f'{metric} ({unit})')
    for c in CLASSES:
        s = d[d.protein_class == c]
        axes[1].scatter(s.n_res, s[metric], s=4, alpha=0.4, color=CLASS_COLOR[c], label=CLASS_LABEL[c])
    axes[1].set_xscale('log'); axes[1].set_xlabel('residues'); axes[1].set_ylabel(metric); axes[1].legend(fontsize=7, markerscale=3)
    fig.tight_layout(); plt.show()
    display(A.structural_contrasts(sh, unit, [metric]).round(3))

metric_view(METRIC, UNIT)

## Protein lookup

In [ ]:
def protein_view(gene):
    hit = prot[prot.gene.str.upper() == gene.upper()]
    if hit.empty: print('no protein with gene', gene); return
    acc = hit.accession.iat[0]
    display(hit.T)
    a = D['protein_auc']; display(a[a.accession == acc][['predictor', 'n_pathogenic', 'n_benign', 'auc', 'ci_low', 'ci_high']].round(3))
    v = lab_all[lab_all.accession == acc]
    if len(v):
        p = PREDICTOR if PREDICTOR in v else PRED[0]
        fig, ax = plt.subplots(figsize=(10, 3))
        for cl, col in (('pathogenic', '#e34948'), ('likely_pathogenic', '#e87ba4'), ('vus', '#8f8d85'), ('likely_benign', '#1baf7a'), ('benign', '#2a78d6')):
            s = v[v.clinvar_class == cl]; ax.scatter(s.position, s[p] * DIRECTION[p], s=8, color=col, label=cl)
        for r in D['residues'][(D['residues'].accession == acc) & D['residues'].in_tm].itertuples():
            ax.axvspan(r.position - 0.5, r.position + 0.5, color='#f3e6dc', lw=0, zorder=0)
        ax.set_xlabel('residue (shaded: TM segments)'); ax.set_ylabel(P.label(p) + ' (oriented)'); ax.legend(fontsize=7, ncol=5)
        ax.set_title(f'{gene} ({acc})', loc='left'); plt.show()

protein_view(GENE)

## Interactive (optional)
With `ipywidgets` installed (`pip install ipywidgets`), dropdowns re-draw the views.

In [ ]:
try:
    import ipywidgets as W
    W.interact(predictor_view, p=W.Dropdown(options=PRED, value=PREDICTOR))
    W.interact(metric_view, metric=W.Dropdown(options=SHAPE_METRICS, value=METRIC), unit=W.Dropdown(options=['CHAIN_FULL', 'CHAIN_TRIMMED', 'DOMAIN', 'TM_CORE'], value=UNIT))
    W.interact(protein_view, gene=W.Text(value=GENE))
except ImportError:
    print('ipywidgets not installed: edit the parameters above and re-run instead')